# Inspect a saved MSC barycenter bundle

Load a complete barycenter folder written by `msc_barycenter_shortestpath.ipynb` or `msc_barycenter_lambda_converge.ipynb` and print every computation parameter recorded in `meta.json`.

The catalog is the **union** of knobs used by both notebooks. Parameters that a given run never wrote (for example `lambda_init` on a shortest-path bundle, or older folders saved before this catalog existed) are listed as **MISSING** rather than guessed.

Point `msc_barycenter_evaluation.ipynb` or `msc_barycenter_reconstruction.ipynb` at the same `BARYCENTER_OUTDIR` when you want to reuse a specific timestamped run (`<keyword>_<mm-dd-yyyy-hh-mm-ss>` or `<keyword>_lambda_converge_<mm-dd-yyyy-hh-mm-ss>`).


In [6]:
import importlib
from pathlib import Path

import pandas as pd

import lib.barycenter_io as barycenter_io

importlib.reload(barycenter_io)
from lib.barycenter_io import (
    format_bundle_parameter_report,
    inspect_bundle_parameters,
    known_bundle_param_keys,
    list_barycenter_bundles,
    load_barycenter_bundle,
    print_barycenter_bundle_menu,
    select_barycenter_bundle,
)


## Available bundles

Set `DATA_KEYWORD` to filter one dataset. Rows are newest-first; legacy `*_latest` folders still appear. Use `BUNDLE_INDEX` or `BUNDLE_RUN_NAME` in the next cell to load one of them.


In [12]:
DATA_KEYWORD = "synthetic"
# DATA_KEYWORD = "VortexStreet"
BUNDLE_PIPELINE = None  # None | "shortestpath" | "lambda_converge"

available = list_barycenter_bundles(keyword=DATA_KEYWORD, pipeline=BUNDLE_PIPELINE)
print_barycenter_bundle_menu(
    available, keyword=DATA_KEYWORD, pipeline=BUNDLE_PIPELINE
)
if not available:
    print("No saved bundles under results/barycenters/")
else:
    display(
        pd.DataFrame(
            [
                {
                    "index": i,
                    "run_name": row["run_name"],
                    "pipeline": row["pipeline"],
                    "saved_at": row["saved_at"],
                    "source_notebook": row["source_notebook"],
                    "DATA_KEYWORD": row["DATA_KEYWORD"],
                    "n_inputs": row["n_inputs"],
                    "n_center": row["n_center"],
                }
                for i, row in enumerate(available)
            ]
        )
    )


Bundles (keyword='synthetic'): 5
  [0] synthetic_lambda_converge_08-31-2026-16-51-54
      pipeline=lambda_converge  saved_at=08-31-2026-16-51-54  n_inputs=9  n_center=427
  [1] synthetic_08-31-2026-01-48-28
      pipeline=shortestpath  saved_at=08-31-2026-01-48-28  n_inputs=9  n_center=423
  [2] synthetic_lambda_converge_08-30-2026-20-33-07
      pipeline=lambda_converge  saved_at=08-30-2026-20-33-07  n_inputs=9  n_center=427
  [3] synthetic_lambda_converge_latest
      pipeline=lambda_converge  saved_at=None  n_inputs=9  n_center=424
  [4] synthetic_latest
      pipeline=shortestpath  saved_at=None  n_inputs=9  n_center=423


,index,run_name,pipeline,saved_at,source_notebook,DATA_KEYWORD,n_inputs,n_center
0,0,synthetic_lambda_converge_08-31-2026-16-51-54,lambda_converge,08-31-2026-16-51-54,msc_barycenter_lambda_converge.ipynb,synthetic,9,427
1,1,synthetic_08-31-2026-01-48-28,shortestpath,08-31-2026-01-48-28,msc_barycenter_shortestpath.ipynb,synthetic,9,423
2,2,synthetic_lambda_converge_08-30-2026-20-33-07,lambda_converge,08-30-2026-20-33-07,msc_barycenter_lambda_converge.ipynb,synthetic,9,427
3,3,synthetic_lambda_converge_latest,lambda_converge,NaN,NaN,NaN,9,424
4,4,synthetic_latest,shortestpath,NaN,NaN,NaN,9,423


## Load one bundle

Set `BUNDLE_INDEX` to a row from the table (0 = newest), or set `BUNDLE_RUN_NAME` to an exact folder name.


In [8]:
BUNDLE_INDEX = 0
BUNDLE_RUN_NAME = None  # e.g. "synthetic_08-29-2026-19-15-32"; overrides index

selected = select_barycenter_bundle(
    DATA_KEYWORD,
    index=BUNDLE_INDEX,
    run_name=BUNDLE_RUN_NAME,
    pipeline=BUNDLE_PIPELINE,
)
BARYCENTER_OUTDIR = selected["path"]
bundle = load_barycenter_bundle(BARYCENTER_OUTDIR)
meta = bundle["meta"]
print(
    f"Using bundle [{selected.get('index')}] {selected['run_name']} "
    f"-> {Path(BARYCENTER_OUTDIR).resolve()}"
)
print(
    f"C_center {tuple(bundle['C_center'].shape)}, "
    f"X_center {tuple(bundle['X_center'].shape)}, "
    f"n_inputs={len(bundle['Cs'])}"
)


Using bundle [0] synthetic_lambda_converge_08-30-2026-20-33-07 -> C:\Users\mingz\desktop\MSCBarycenter\results\barycenters\synthetic_lambda_converge_08-30-2026-20-33-07
C_center (427, 427), X_center (427, 8), n_inputs=9


## All known parameters

Every catalog key is requested from the saved `meta`. Missing keys are those this bundle does not contain — including knobs that belong only to the other computation notebook.


In [9]:
inspection = inspect_bundle_parameters(meta, outdir=BARYCENTER_OUTDIR)
print(format_bundle_parameter_report(inspection))


Bundle: C:\Users\mingz\desktop\MSCBarycenter\results\barycenters\synthetic_lambda_converge_08-30-2026-20-33-07
pipeline: lambda_converge
source_notebook: msc_barycenter_lambda_converge.ipynb
saved_at: 08-30-2026-20-33-07
catalog: 72 present, 0 missing (of 72)

=== run ===
  saved_at                             "08-30-2026-20-33-07"
  source_notebook                      "msc_barycenter_lambda_converge.ipynb"
  pipeline                             "lambda_converge"
  DATA_ROOT                            "data"
  DATA_KEYWORD                         "synthetic"
  MSC_DIR                              "data\\synthetic\\DESC-MC_0.0"
  configured_mesh_ids                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

=== sampling ===
  SAMPLE_RATE                          0.02
  SAMPLE_MODE                          "geo"
  SAMPLE_COMPONENT                     "largest"

=== seed ===
  seed_graph_id                        0
  seed_mesh_id                         0
  exclude_seed                         tru

In [10]:
rows = []
for key in known_bundle_param_keys():
    present = key in meta
    rows.append(
        {
            "parameter": key,
            "status": "present" if present else "MISSING",
            "value": meta[key] if present else None,
        }
    )
param_table = pd.DataFrame(rows)
display(param_table)

missing = param_table.loc[param_table["status"] == "MISSING", "parameter"].tolist()
print(f"\nMissing parameters ({len(missing)} of {len(param_table)}):")
if missing:
    for key in missing:
        print(f"  - {key}")
else:
    print("  (none)")

extra = inspection["extra_keys"]
print(f"\nExtra keys in meta.json not in the catalog ({len(extra)}):")
if extra:
    for key in extra:
        print(f"  - {key} = {meta[key]!r}")
else:
    print("  (none)")


,parameter,status,value
0,saved_at,present,08-30-2026-20-33-07
1,source_notebook,present,msc_barycenter_lambda_converge.ipynb
2,pipeline,present,lambda_converge
3,DATA_ROOT,present,data
4,DATA_KEYWORD,present,synthetic
...,...,...,...
67,has_Ts,present,True
68,has_G_complete,present,True
69,has_G_mst,present,True
70,has_input_graphs,present,True



Missing parameters (0 of 72):
  (none)

Extra keys in meta.json not in the catalog (0):
  (none)
